# Run-to-run reproducibility of the measured table reference

This notebook compares the measured shaking-table responses from the paired nominal 30- and 50-Hz acquisition runs for each earthquake.

**Input:** the 14 retained TDMS reference files under `data/`.

**Outputs:** event-level reproducibility metrics, paired amplitude tests, and the summary table under `results/`.


In [ ]:
%pip -q install nptdms

from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
from scipy import signal, stats
from nptdms import TdmsFile

cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data directory not found: {DATA_DIR}\n"
        "Create a 'data' directory in the repository and place the experimental data there."
    )

for archive in DATA_DIR.glob("*.zip"):
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(DATA_DIR)

RUNS = {
    "Bingol":    {30: "111445", 50: "111811"},
    "Kirikhan":  {30: "112036", 50: "112343"},
    "Sindirgi":  {30: "112705", 50: "112951"},
    "Sulusaray": {30: "113156", 50: "113737"},
    "Bozkurt":   {30: "114254", 50: "114537"},
    "Goksun":    {30: "114901", 50: "115059"},
    "Van":       {30: "115302", 50: "115537"},
}

FMIN, FMAX = 0.5, 12.0
MAX_ALIGNMENT_LAG_S = 5.0
BASELINE_SECONDS = 5.0
ACTIVE_THRESHOLD = 0.02
ACTIVE_PAD_SECONDS = 2.0

tdms_files = [p for p in DATA_DIR.rglob("*.tdms") if p.is_file()]

def find_tdms(run_id):
    matches = [p for p in tdms_files if run_id in p.name]
    if len(matches) != 1:
        raise RuntimeError(f"TDMS run {run_id}: expected one match, found {len(matches)}")
    return matches[0]

for event in RUNS:
    for setting in (30, 50):
        find_tdms(RUNS[event][setting])

print("All 14 retained TDMS reference files were found.")


In [ ]:
def get_channel(tdms, name):
    found = []
    for group in tdms.groups():
        for channel in group.channels():
            if channel.name.strip().lower() == name.lower():
                found.append(channel)
    if len(found) != 1:
        raise RuntimeError(f"Channel {name}: expected one match, found {len(found)}")
    return found[0]

def load_reference(path):
    tdms = TdmsFile.read(path)
    ch1 = get_channel(tdms, "Acceleration-1")
    ch2 = get_channel(tdms, "Acceleration-2")

    x = np.asarray(ch1[:], dtype=float)
    y = np.asarray(ch2[:], dtype=float)
    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    fs1 = 1.0 / float(ch1.properties.get("wf_increment", 1 / 102.4))
    fs2 = 1.0 / float(ch2.properties.get("wf_increment", 1 / 102.4))
    if abs(fs1 - fs2) > 1e-6:
        raise RuntimeError(f"Reference sampling rates do not match: {fs1} vs {fs2}")

    return x, y, fs1

def baseline_correct(x, fs, seconds=5.0):
    n0 = min(len(x), max(1, int(round(seconds * fs))))
    return x - np.median(x[:n0])

def horizontal_resultant(x, y):
    return np.sqrt(x**2 + y**2)

def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    s = np.std(x)
    return (x - np.mean(x)) / s if s >= 1e-12 else x * 0.0

def estimate_lag(a, b, fs, max_lag_s=5.0):
    aa = zscore_safe(a)
    bb = zscore_safe(b)
    corr = signal.correlate(bb, aa, mode="full", method="fft")
    lags = signal.correlation_lags(len(bb), len(aa), mode="full")
    keep = np.abs(lags) <= int(round(max_lag_s * fs))
    return int(lags[keep][np.argmax(corr[keep])])

def overlap_indices(n_a, n_b, lag):
    if lag >= 0:
        start_a, start_b = 0, lag
    else:
        start_a, start_b = -lag, 0
    n = min(n_a - start_a, n_b - start_b)
    if n <= 10:
        raise RuntimeError("Insufficient overlap after alignment")
    return start_a, start_b, n

def active_window(h30, h50, fs, threshold=0.02, pad_s=2.0):
    if np.max(h30) <= 0 or np.max(h50) <= 0:
        return 0, min(len(h30), len(h50))

    activity = np.maximum(h30 / np.max(h30), h50 / np.max(h50))
    idx = np.flatnonzero(activity >= threshold)
    if len(idx) == 0:
        return 0, min(len(h30), len(h50))

    pad = int(round(pad_s * fs))
    return max(0, idx[0] - pad), min(len(activity), idx[-1] + pad + 1)

def pearson_safe(a, b):
    if len(a) < 3 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def vector_psd(x, y, fs):
    nperseg = min(len(x), max(8, int(round(fs * 8.0))))
    f, px = signal.welch(x, fs=fs, nperseg=nperseg, detrend="constant")
    _, py = signal.welch(y, fs=fs, nperseg=nperseg, detrend="constant")
    return f, px + py

def spectral_metrics(x30, y30, x50, y50, fs):
    f30, p30 = vector_psd(x30, y30, fs)
    f50, p50 = vector_psd(x50, y50, fs)
    grid = np.linspace(FMIN, FMAX, 500)
    p30i = np.interp(grid, f30, p30)
    p50i = np.interp(grid, f50, p50)

    eps = 1e-20
    psd_corr = pearson_safe(np.log10(p30i + eps), np.log10(p50i + eps))
    dom30 = float(grid[np.argmax(p30i)])
    dom50 = float(grid[np.argmax(p50i)])

    trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    centroid30 = float(trapz(grid * p30i, grid) / trapz(p30i, grid))
    centroid50 = float(trapz(grid * p50i, grid) / trapz(p50i, grid))

    bands = [(0.5, 3.0), (3.0, 6.0), (6.0, 9.0), (9.0, 12.0)]

    def fractions(p):
        values = []
        for lo, hi in bands:
            mask = (grid >= lo) & (grid <= hi)
            values.append(trapz(p[mask], grid[mask]))
        values = np.asarray(values, dtype=float)
        return values / values.sum()

    bf30 = fractions(p30i)
    bf50 = fractions(p50i)

    return {
        "psd_corr": psd_corr,
        "dom_freq_30_Hz": dom30,
        "dom_freq_50_Hz": dom50,
        "dom_freq_abs_diff_Hz": abs(dom50 - dom30),
        "centroid_30_Hz": centroid30,
        "centroid_50_Hz": centroid50,
        "centroid_abs_diff_Hz": abs(centroid50 - centroid30),
        "band_energy_L1": float(np.sum(np.abs(bf30 - bf50))),
    }


In [ ]:
rows = []

for event, runs in RUNS.items():
    x30, y30, fs30 = load_reference(find_tdms(runs[30]))
    x50, y50, fs50 = load_reference(find_tdms(runs[50]))

    if abs(fs30 - fs50) > 1e-6:
        raise RuntimeError(f"{event}: sampling-rate mismatch")
    fs = fs30

    x30 = baseline_correct(x30, fs, BASELINE_SECONDS)
    y30 = baseline_correct(y30, fs, BASELINE_SECONDS)
    x50 = baseline_correct(x50, fs, BASELINE_SECONDS)
    y50 = baseline_correct(y50, fs, BASELINE_SECONDS)

    h30 = horizontal_resultant(x30, y30)
    h50 = horizontal_resultant(x50, y50)

    lag = estimate_lag(h30, h50, fs, MAX_ALIGNMENT_LAG_S)
    s30, s50, n = overlap_indices(len(h30), len(h50), lag)

    x30a, y30a = x30[s30:s30+n], y30[s30:s30+n]
    x50a, y50a = x50[s50:s50+n], y50[s50:s50+n]
    h30a = horizontal_resultant(x30a, y30a)
    h50a = horizontal_resultant(x50a, y50a)

    i0, i1 = active_window(
        h30a, h50a, fs,
        threshold=ACTIVE_THRESHOLD,
        pad_s=ACTIVE_PAD_SECONDS,
    )

    x30w, y30w = x30a[i0:i1], y30a[i0:i1]
    x50w, y50w = x50a[i0:i1], y50a[i0:i1]
    h30w = horizontal_resultant(x30w, y30w)
    h50w = horizontal_resultant(x50w, y50w)

    pga30, pga50 = float(np.max(h30w)), float(np.max(h50w))
    rms30 = float(np.sqrt(np.mean(h30w**2)))
    rms50 = float(np.sqrt(np.mean(h50w**2)))
    pga_ratio, rms_ratio = pga50 / pga30, rms50 / rms30

    rows.append({
        "event": event,
        "run30": runs[30],
        "run50": runs[50],
        "fs_table_Hz": fs,
        "alignment_lag_samples": lag,
        "alignment_lag_s": lag / fs,
        "analysis_duration_s": len(h30w) / fs,
        "h_corr_30_vs_50": pearson_safe(h30w - np.mean(h30w), h50w - np.mean(h50w)),
        "PGA_30_g": pga30,
        "PGA_50_g": pga50,
        "PGA_ratio_50_30": pga_ratio,
        "PGA_change_pct": 100 * (pga_ratio - 1),
        "RMS_30_g": rms30,
        "RMS_50_g": rms50,
        "RMS_ratio_50_30": rms_ratio,
        "RMS_change_pct": 100 * (rms_ratio - 1),
        **spectral_metrics(x30w, y30w, x50w, y50w, fs),
    })

run_reproducibility = pd.DataFrame(rows)
event_order = ["Bingol", "Bozkurt", "Goksun", "Kirikhan", "Sindirgi", "Sulusaray", "Van"]
run_reproducibility["event"] = pd.Categorical(
    run_reproducibility["event"], categories=event_order, ordered=True
)
run_reproducibility = run_reproducibility.sort_values("event").reset_index(drop=True)

display(
    run_reproducibility[
        [
            "event", "h_corr_30_vs_50", "PGA_change_pct", "RMS_change_pct",
            "psd_corr", "dom_freq_abs_diff_Hz", "centroid_abs_diff_Hz",
            "band_energy_L1",
        ]
    ].round(4)
)


In [ ]:
def paired_log_test(ratio, name):
    ratio = np.asarray(ratio, dtype=float)
    log_ratio = np.log(ratio)
    t = stats.ttest_1samp(log_ratio, 0.0)

    try:
        w = stats.wilcoxon(log_ratio, alternative="two-sided", method="auto")
        w_stat, w_p = w.statistic, w.pvalue
    except Exception:
        w_stat, w_p = np.nan, np.nan

    sd = np.std(log_ratio, ddof=1)
    dz = np.mean(log_ratio) / sd if sd > 0 else np.nan

    return {
        "measure": name,
        "median_ratio_50_30": np.median(ratio),
        "median_change_pct": 100 * (np.median(ratio) - 1),
        "median_abs_change_pct": np.median(np.abs(100 * (ratio - 1))),
        "mean_change_pct": np.mean(100 * (ratio - 1)),
        "t_logratio": t.statistic,
        "p_t_logratio": t.pvalue,
        "wilcoxon_stat": w_stat,
        "p_wilcoxon": w_p,
        "cohen_dz_logratio": dz,
    }

amplitude_tests = pd.DataFrame([
    paired_log_test(run_reproducibility["PGA_ratio_50_30"], "Horizontal PGA"),
    paired_log_test(run_reproducibility["RMS_ratio_50_30"], "Horizontal RMS"),
])

similarity_summary = pd.DataFrame({
    "metric": [
        "Horizontal-resultant correlation",
        "Vector-PSD correlation",
        "Dominant-frequency absolute difference (Hz)",
        "Spectral-centroid absolute difference (Hz)",
        "Band-energy L1 distance",
    ],
    "median": [
        run_reproducibility["h_corr_30_vs_50"].median(),
        run_reproducibility["psd_corr"].median(),
        run_reproducibility["dom_freq_abs_diff_Hz"].median(),
        run_reproducibility["centroid_abs_diff_Hz"].median(),
        run_reproducibility["band_energy_L1"].median(),
    ],
    "min": [
        run_reproducibility["h_corr_30_vs_50"].min(),
        run_reproducibility["psd_corr"].min(),
        run_reproducibility["dom_freq_abs_diff_Hz"].min(),
        run_reproducibility["centroid_abs_diff_Hz"].min(),
        run_reproducibility["band_energy_L1"].min(),
    ],
    "max": [
        run_reproducibility["h_corr_30_vs_50"].max(),
        run_reproducibility["psd_corr"].max(),
        run_reproducibility["dom_freq_abs_diff_Hz"].max(),
        run_reproducibility["centroid_abs_diff_Hz"].max(),
        run_reproducibility["band_energy_L1"].max(),
    ],
})

run_reproducibility.to_csv(
    RESULTS_DIR / "TDMS_30vs50_RunReproducibility.csv", index=False
)
amplitude_tests.to_csv(
    RESULTS_DIR / "TDMS_30vs50_AmplitudeTests.csv", index=False
)
similarity_summary.to_csv(
    RESULTS_DIR / "TDMS_30vs50_SimilaritySummary.csv", index=False
)

display(amplitude_tests.round(4))
display(similarity_summary.round(4))
